## STEP 1 — Setup


In [1]:
\
import os, re, ast, json, pickle, random
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
from surprise import Dataset, Reader, SVD, accuracy

SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# DATA_DIR: env override > current directory
DATA_DIR = Path(os.environ.get("FILMMOSAIC_DATA_DIR", Path.cwd()))
ARTIFACT_DIR = Path(os.environ.get("FILMMOSAIC_ARTIFACT_DIR", DATA_DIR))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
print("DATA_DIR    =", DATA_DIR)
print("ARTIFACT_DIR=", ARTIFACT_DIR)
print("✅ Step 1 done")


DATA_DIR    = /Users/johnshrestha/Documents/Recome
ARTIFACT_DIR= /Users/johnshrestha/Documents/Recome
✅ Step 1 done


## STEP 2 — Load + clean


In [2]:
\
movies        = pd.read_csv(DATA_DIR / "combined_movies.csv")
ratings       = pd.read_csv(DATA_DIR / "synthetic_movie_ratings_realistic.csv")
interactions  = pd.read_csv(DATA_DIR / "synthetic_movie_interactions_realistic.csv")
user_profiles = pd.read_csv(DATA_DIR / "synthetic_movie_user_profiles.csv")

movies = movies.drop_duplicates(subset="movie_id", keep="first").reset_index(drop=True)
movies["is_regional"] = movies["is_regional"].fillna(False).astype(bool)

# Frozen reference date — reproducible trending scores
ref_date = pd.Timestamp("2026-06-01")

for df in (ratings, interactions):
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df.dropna(subset=["timestamp"], inplace=True)
    df["age_days"] = (ref_date - df["timestamp"]).dt.days.clip(lower=0)

ratings = ratings[ratings["rating"].between(0.5, 5.0)].reset_index(drop=True)

# Union of rated + interacted (this is the master "watched" set)
watched_pairs = pd.concat([
    ratings[["user_id", "movie_id"]],
    interactions[["user_id", "movie_id"]],
], ignore_index=True).drop_duplicates()

print(f"Movies       : {movies.shape}  regional={int(movies['is_regional'].sum())}")
print(f"Ratings      : {ratings.shape}  users={ratings['user_id'].nunique()}")
print(f"Interactions : {interactions.shape}  types={interactions['interaction_type'].unique().tolist()}")
print(f"User profiles: {user_profiles.shape}  archetypes={user_profiles['archetype'].nunique()}")
print(f"Watched pairs (union): {len(watched_pairs):,}")
print(f"Reference date (frozen): {ref_date.date()}")
print("✅ Step 2 done")


Movies       : (5119, 8)  regional=443
Ratings      : (6864, 6)  users=240
Interactions : (9542, 11)  types=['watchlist', 'rating', 'view', 'review']
User profiles: (240, 4)  archetypes=8
Watched pairs (union): 9,542
Reference date (frozen): 2026-06-01
✅ Step 2 done


## STEP 3 — Genre / language helpers


In [3]:
\
def safe_literal(x):
    """Parse a genre cell that may be a Python-list string, JSON, or pipe-joined."""
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    for parser in (ast.literal_eval, json.loads):
        try:
            v = parser(s)
            if isinstance(v, (list, tuple)):
                return [str(t).strip() for t in v if str(t).strip()]
        except Exception:
            pass
    items = re.findall(r"'([^']+)'|\"([^\"]+)\"", s)
    flat = [a or b for a, b in items]
    if flat:
        return [t.strip() for t in flat if t.strip()]
    return [t.strip() for t in re.split(r"[|,/]", s) if t.strip()]

def tokenize_genres(x):
    return {g.lower() for g in safe_literal(x)}

def tokenize_pipe(x):
    if pd.isna(x):
        return set()
    return {g.strip().lower() for g in str(x).split("|") if g.strip()}

def to_genre_string(x):
    return " ".join(sorted(tokenize_genres(x)))

movies["genre_list"]   = movies["genres"].apply(safe_literal)
movies["genre_set"]    = movies["genres"].apply(tokenize_genres)
movies["keyword_list"] = movies["keywords"].apply(safe_literal)
movies["keyword_set"]  = movies["keywords"].apply(tokenize_genres)
movies["genres_clean"] = movies["genres"].apply(to_genre_string)

user_profiles["archetype"] = (
    user_profiles["archetype"].fillna("mixed_viewer").astype(str).str.lower().str.strip()
)
user_profiles["preferred_genres"] = user_profiles["preferred_genres"].fillna("").astype(str)
user_profiles["pref_set"] = user_profiles["preferred_genres"].apply(tokenize_pipe)
user_profiles["interaction_target"] = pd.to_numeric(
    user_profiles["interaction_target"], errors="coerce"
).fillna(user_profiles["interaction_target"].median())

profile_lookup = user_profiles.set_index("user_id").to_dict("index")
movie_lookup   = movies.set_index("movie_id").to_dict("index")

ARCHETYPE_BOOSTS = {
    "sci_fi_fantasy":   {"science fiction", "fantasy", "adventure", "action"},
    "horror_thriller":  {"horror", "thriller", "mystery"},
    "comedy_family":    {"comedy", "family", "animation"},
    "classic_drama":    {"drama", "history", "war"},
    "crime_mystery":    {"crime", "mystery", "thriller"},
    "mainstream_action":{"action", "adventure", "thriller"},
    "drama_romance":    {"drama", "romance"},
    "mixed_viewer":     set(),
}
print("Archetypes:", sorted(user_profiles["archetype"].unique()))
print("Sample genres parsed:", movies["genre_list"].head(3).tolist())
print("✅ Step 3 done")


Archetypes: ['classic_drama', 'comedy_family', 'crime_mystery', 'drama_romance', 'horror_thriller', 'mainstream_action', 'mixed_viewer', 'sci_fi_fantasy']
Sample genres parsed: [['Action', 'Science Fiction', 'Thriller'], ['Action', 'Crime', 'Thriller'], ['Horror', 'Mystery', 'Crime']]
✅ Step 3 done


## STEP 4 — SBERT content model (fixed text)


In [4]:
\
MODEL_NAME = "all-MiniLM-L6-v2"
print("Loading SBERT:", MODEL_NAME)
sbert = SentenceTransformer(MODEL_NAME)

def build_sbert_text(row):
    parts = []
    if row["title"]:
        parts.append(f"Title: {row['title']}.")
    if row["genres_clean"]:
        parts.append(f"Genres: {row['genres_clean']}.")
    if isinstance(row["keywords"], str) and row["keywords"].strip():
        parts.append(f"Keywords: {row['keywords'].replace('|', ' ')}.")
    if isinstance(row["overview"], str) and row["overview"].strip():
        parts.append(f"Overview: {row['overview']}.")
    if row.get("language"):
        parts.append(f"Language: {row['language']}.")
    if row.get("is_regional"):
        parts.append("Regional film.")
    return " ".join(parts)

movies["sbert_text"] = movies.apply(build_sbert_text, axis=1)

texts = movies["sbert_text"].fillna("").tolist()
print("Encoding", len(texts), "movies...")
movie_embeddings = sbert.encode(
    texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
print("Embedding shape:", movie_embeddings.shape)

with open(ARTIFACT_DIR / "embeddings.pkl", "wb") as f:
    pickle.dump(movie_embeddings, f)
print("✅ Step 4 done — embeddings.pkl saved")


Loading SBERT: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 5119 movies...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Embedding shape: (5119, 384)
✅ Step 4 done — embeddings.pkl saved


## STEP 5 — Content similarity matrix


In [5]:
\
content_similarity = cosine_similarity(movie_embeddings)
np.fill_diagonal(content_similarity, 0.0)

with open(ARTIFACT_DIR / "similarity.pkl", "wb") as f:
    pickle.dump(content_similarity, f)

print("Shape:", content_similarity.shape)
nz = content_similarity[content_similarity > 0]
print(f"Non-zero off-diag: {nz.size:,}  min={nz.min():.4f}  max={nz.max():.4f}  mean={nz.mean():.4f}")
print("✅ Step 5 done — similarity.pkl saved")


Shape: (5119, 5119)
Non-zero off-diag: 26,178,802  min=0.0000  max=0.9767  mean=0.3004
✅ Step 5 done — similarity.pkl saved


## STEP 6 — SVD collaborative filtering


In [6]:
\
def train_svd(train_df, n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.05):
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(
        train_df[["user_id", "movie_id", "rating"]], reader
    )
    trainset = data.build_full_trainset()
    model = SVD(n_factors=n_factors, n_epochs=n_epochs,
                lr_all=lr_all, reg_all=reg_all, random_state=SEED)
    model.fit(trainset)
    return model

def predict_score(model, uid, iid):
    pred = model.predict(uid=uid, iid=iid).est
    return float(np.clip((pred - 0.5) / 4.5, 0.0, 1.0))

cf_model = train_svd(ratings)
print("✅ Step 6 done — SVD trained on", len(ratings), "ratings")


✅ Step 6 done — SVD trained on 6864 ratings


## STEP 7 — Trending component


In [7]:
\
recent = interactions.copy()
recent["recency_w"] = 1.0 / (1.0 + recent["age_days"])

recent_score    = recent.groupby("movie_id")["recency_w"].sum().rename("recent_score").reset_index()
rating_activity = ratings.groupby("movie_id").size().rename("rating_count").reset_index()
avg_rating      = ratings.groupby("movie_id")["rating"].mean().rename("avg_rating").reset_index()

trending_df = movies[["movie_id"]].copy()
trending_df = trending_df.merge(recent_score,    on="movie_id", how="left")
trending_df = trending_df.merge(rating_activity, on="movie_id", how="left")
trending_df = trending_df.merge(avg_rating,      on="movie_id", how="left")
trending_df[["recent_score", "rating_count", "avg_rating"]] = (
    trending_df[["recent_score", "rating_count", "avg_rating"]].fillna(0)
)

cols = ["recent_score", "rating_count", "avg_rating"]
trending_df[cols] = MinMaxScaler().fit_transform(trending_df[cols])

trending_df["trending_score"] = (
    0.60 * trending_df["recent_score"]
    + 0.25 * trending_df["rating_count"]
    + 0.15 * trending_df["avg_rating"]
)
trending_df["trending_score"] = MinMaxScaler().fit_transform(trending_df[["trending_score"]])
trending_scores = dict(zip(trending_df["movie_id"], trending_df["trending_score"]))

with open(ARTIFACT_DIR / "trending.pkl", "wb") as f:
    pickle.dump(trending_scores, f)
print(f"Trending scored: {len(trending_scores):,} movies")
print("✅ Step 7 done — trending.pkl saved")


Trending scored: 5,119 movies
✅ Step 7 done — trending.pkl saved


## STEP 8 — Underrated & regional scoring


In [8]:
\
# Bayesian-shrunk mean rating (IMDb formula):
#   score = (v / (v + m)) * R + (m / (v + m)) * C
C = float(ratings["rating"].mean())
m = float(ratings["movie_id"].value_counts().quantile(0.75))

vc = ratings["movie_id"].value_counts().rename("votes")
rm = ratings.groupby("movie_id")["rating"].mean().rename("mean_r")
bayes = vc.to_frame().join(rm).fillna({"votes": 0, "mean_r": C})
bayes["underrated_score"] = (
    (bayes["votes"] / (bayes["votes"] + m)) * bayes["mean_r"]
    + (m          / (bayes["votes"] + m)) * C
)
u_min, u_max = bayes["underrated_score"].min(), bayes["underrated_score"].max()
bayes["underrated_score"] = (bayes["underrated_score"] - u_min) / (u_max - u_min + 1e-9)
underrated_scores = bayes["underrated_score"].to_dict()

regional_scores = movies.set_index("movie_id")["is_regional"].astype(int).to_dict()

with open(ARTIFACT_DIR / "underrated.pkl", "wb") as f:
    pickle.dump(underrated_scores, f)
with open(ARTIFACT_DIR / "regional.pkl", "wb") as f:
    pickle.dump(regional_scores, f)
print(f"Underrated: m={m:.1f}  C={C:.3f}  scored {len(underrated_scores):,}")
print(f"Regional flag: {sum(regional_scores.values()):,} movies")
print("✅ Step 8 done")


Underrated: m=3.0  C=3.624  scored 3,321
Regional flag: 443 movies
✅ Step 8 done


## STEP 9 — Profile / archetype matching


In [9]:
\
def profile_score_for(user_id, movie_id):
    profile = profile_lookup.get(user_id)
    if not profile:
        return 0.0
    pref  = profile.get("pref_set") or set()
    boost = ARCHETYPE_BOOSTS.get(profile.get("archetype", "mixed_viewer"), set())
    if not pref and not boost:
        return 0.0
    row = movie_lookup.get(movie_id, {})
    movie_g = row.get("genre_set") or set()
    if not movie_g:
        return 0.0
    jacc = (len(pref & movie_g) / len(pref | movie_g)) if (pref | movie_g) else 0.0
    arc  = (len(boost & movie_g) / len(boost)) if boost else 0.0
    target = float(profile.get("interaction_target", 40))
    target_factor = float(np.clip(target / 40.0, 0.75, 1.25))
    return float(np.clip((0.75 * jacc + 0.25 * arc) * target_factor, 0.0, 1.0))

# Precompute the (user × movie) profile matrix once
profile_matrix = pd.DataFrame(
    {uid: {mid: profile_score_for(uid, mid) for mid in movies["movie_id"]}
     for uid in user_profiles["user_id"]}
)
print("profile_matrix:", profile_matrix.shape)
print("✅ Step 9 done")


profile_matrix: (5119, 240)
✅ Step 9 done


## STEP 10 — Popularity debiasing (off by default, opt-in for diversity)


In [10]:
\
# Popularity debiasing was a v1 leftover. Empirical sweep on this dataset
# (alpha 0.0 .. 0.2) shows it *hurts* ranking quality — the held-out positives
# are, by construction, popular. We expose it as an opt-in flag for the
# `recommend()` call when you want to actively diversify the slate.
movie_interaction_counts = interactions.groupby("movie_id").size().to_dict()

def popularity_penalty(movie_id, alpha=0.10):
    if alpha <= 0:
        return 1.0
    c = movie_interaction_counts.get(movie_id, 0)
    return 1.0 + alpha * np.log(1.0 + c)

def debias(score, movie_id, alpha=0.0):
    if not score or alpha <= 0:
        return float(score or 0.0)
    return float(score) / popularity_penalty(movie_id, alpha)

print("Movies with interaction counts:", len(movie_interaction_counts))
print("Max interactions:", max(movie_interaction_counts.values(), default=0))
print("✅ Step 10 done (debias available, but OFF by default — see recommend())")


Movies with interaction counts: 3733
Max interactions: 15
✅ Step 10 done (debias available, but OFF by default — see recommend())


## STEP 11 — Hybrid engine (adaptive + time decay + slate diversity)


In [11]:
\
movie_id_to_idx = {mid: i for i, mid in enumerate(movies["movie_id"])}
idx_to_movie_id = {i: mid for mid, i in movie_id_to_idx.items()}
movie_ids_arr   = movies["movie_id"].to_numpy()

def user_watch_history(user_id):
    r = ratings.loc[ratings["user_id"] == user_id, ["movie_id", "age_days", "rating"]]
    i = interactions.loc[interactions["user_id"] == user_id, ["movie_id", "age_days"]]
    i = i.assign(rating=np.nan)
    return pd.concat([r, i], ignore_index=True).drop_duplicates(subset="movie_id")

# Per-archetype adaptive weights (sum to 1.0)
ARCHETYPE_WEIGHTS = {
    "horror_thriller":  dict(content=0.25, cf=0.30, profile=0.40, trending=0.05),
    "crime_mystery":    dict(content=0.30, cf=0.30, profile=0.35, trending=0.05),
    "sci_fi_fantasy":   dict(content=0.30, cf=0.30, profile=0.35, trending=0.05),
    "mainstream_action":dict(content=0.25, cf=0.40, profile=0.30, trending=0.05),
    "comedy_family":    dict(content=0.30, cf=0.30, profile=0.30, trending=0.10),
    "classic_drama":    dict(content=0.35, cf=0.30, profile=0.30, trending=0.05),
    "drama_romance":    dict(content=0.30, cf=0.30, profile=0.35, trending=0.05),
    "mixed_viewer":     dict(content=0.30, cf=0.30, profile=0.30, trending=0.10),
}
DEFAULT_WEIGHTS = dict(content=0.30, cf=0.30, profile=0.30, trending=0.10)
COLD_OVERRIDE   = dict(content=0.20, cf=0.05, profile=0.55, trending=0.20)
EARLY_OVERRIDE  = dict(content=0.30, cf=0.20, profile=0.35, trending=0.15)

def get_weights(user_id):
    prof = profile_lookup.get(user_id)
    if not prof:
        return DEFAULT_WEIGHTS
    return ARCHETYPE_WEIGHTS.get(prof["archetype"], DEFAULT_WEIGHTS)

def get_user_type(user_id):
    n = int((watched_pairs["user_id"] == user_id).sum())
    prof = profile_lookup.get(user_id)
    target = float(prof["interaction_target"]) if prof else 40.0
    cold_t, early_t = max(4, int(target * 0.10)), max(29, int(target * 0.60))
    if n <= cold_t:  return "cold_start"
    if n <= early_t: return "early_user"
    return "active_user"

def _normalize_dict(d):
    if not d: return {}
    v = np.array(list(d.values()), dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if np.isclose(lo, hi): return {k: 0.0 for k in d}
    return {k: float((x - lo) / (hi - lo)) for k, x in d.items()}

# ----- content score: vectorized, time-decayed ------------------------------
HALF_LIFE_DAYS = 180.0

def content_score_vector(user_id, candidate_ids):
    hist = user_watch_history(user_id)
    if hist.empty:
        return {mid: 0.0 for mid in candidate_ids}

    # Time decay weight per watched item
    decay = np.exp(-hist["age_days"].to_numpy(dtype=float) / HALF_LIFE_DAYS)

    watched_ids = [m for m in hist["movie_id"] if m in movie_id_to_idx]
    watched_idx = [movie_id_to_idx[m] for m in watched_ids]
    if not watched_idx:
        return {mid: 0.0 for mid in candidate_ids}

    cand_ids = [m for m in candidate_ids if m in movie_id_to_idx]
    cand_idx = [movie_id_to_idx[m] for m in cand_ids]
    if not cand_idx:
        return {mid: 0.0 for mid in candidate_ids}

    sub = content_similarity[np.ix_(watched_idx, cand_idx)]   # (W, C)
    user_vec = (sub.T @ decay) / max(decay.sum(), 1e-9)        # (C,)

    out = {mid: 0.0 for mid in candidate_ids}
    for j, mid in enumerate(cand_ids):
        out[mid] = float(user_vec[j])
    return out

# ----- collaborative score: vectorized over candidates ---------------------
def collaborative_score_vector(user_id, candidate_ids):
    return {mid: predict_score(cf_model, user_id, mid) for mid in candidate_ids}

# ----- profile score: precomputed matrix ------------------------------------
def profile_score_vector(user_id, candidate_ids):
    if user_id not in profile_matrix.columns:
        return {mid: 0.0 for mid in candidate_ids}
    return {mid: float(profile_matrix.at[mid, user_id]) for mid in candidate_ids}

# ----- master hybrid -------------------------------------------------------
def hybrid_score(user_id, candidate_ids, debias_alpha=0.0):
    """Compute hybrid scores. debias_alpha=0.0 (default) is best on this data;
    raise it (e.g. 0.1) only if you want active diversity in the slate."""
    cands = list(candidate_ids)
    cs = _normalize_dict(content_score_vector(user_id, cands))
    fs = _normalize_dict(collaborative_score_vector(user_id, cands))
    ps = _normalize_dict(profile_score_vector(user_id, cands))
    ts = {m: float(trending_scores.get(m, 0.0)) for m in cands}

    # Opt-in popularity debiasing
    if debias_alpha > 0:
        for m in cands:
            fs[m] = debias(fs[m], m, alpha=debias_alpha)
            cs[m] = debias(cs[m], m, alpha=debias_alpha)
        # Re-normalize after debiasing so the weighting stays balanced
        cs = _normalize_dict(cs); fs = _normalize_dict(fs)

    ut = get_user_type(user_id)
    w  = COLD_OVERRIDE if ut == "cold_start" else (EARLY_OVERRIDE if ut == "early_user" else get_weights(user_id))

    final = {}
    for m in cands:
        final[m] = (
            w["content"]  * cs.get(m, 0.0)
          + w["cf"]       * fs.get(m, 0.0)
          + w["profile"]  * ps.get(m, 0.0)
          + w["trending"] * ts.get(m, 0.0)
        )
    return final, cs, fs, ps, ts

print("✅ Step 11 done — hybrid engine ready (debias_alpha=0 by default)")


✅ Step 11 done — hybrid engine ready (debias_alpha=0 by default)


## STEP 12 — Recommend + explain


In [12]:
\
def recommend(user_id, top_n=10, ensure_regional=2, ensure_underrated=1,
              debias_alpha=0.0, watched=None):
    """Top-n recommendations with slate-level diversity injection.

    Parameters
    ----------
    user_id          : user to recommend for
    top_n            : number of items to return
    ensure_regional  : guarantee at least this many regional films in the slate
    ensure_underrated: guarantee at least this many high-avg / low-vote films
    debias_alpha     : 0 = no popularity debiasing (best on this data);
                       raise it to actively diversify the slate
    """
    if watched is None:
        watched = set(watched_pairs.loc[watched_pairs["user_id"] == user_id, "movie_id"])
    cands = [m for m in movie_ids_arr if m not in watched]

    final, cs, fs, ps, ts = hybrid_score(user_id, cands, debias_alpha=debias_alpha)
    ranked = sorted(final.items(), key=lambda x: x[1], reverse=True)

    # Reserve slots for diversity guarantees
    n_regional  = 0
    n_underrate = 0
    if ensure_regional > 0:
        # How many regional items exist in the candidate pool?
        n_regional = min(ensure_regional, sum(1 for m in cands if regional_scores.get(m, 0)))
    if ensure_underrated > 0:
        n_underrate = min(ensure_underrated, sum(
            1 for m in cands if underrated_scores.get(m, 0) > 0.7))

    main_slots = max(top_n - n_regional - n_underrate, 0)

    chosen, chosen_ids = [], set()

    # 1) Fill main slots with top-ranked candidates
    for mid, sc in ranked:
        if len(chosen) >= main_slots: break
        if mid in chosen_ids: continue
        chosen.append((mid, sc)); chosen_ids.add(mid)

    # 2) Reserve regional slots — pick the highest-scoring regional movies
    if n_regional > 0:
        regional_picks = [(m, s) for m, s in ranked
                          if regional_scores.get(m, 0) and m not in chosen_ids]
        regional_picks.sort(key=lambda x: x[1], reverse=True)
        for mid, sc in regional_picks[:n_regional]:
            chosen.append((mid, sc))
            chosen_ids.add(mid)

    # 3) Reserve underrated slots — pick the highest-scoring underrated movies
    if n_underrate > 0:
        und_picks = [(m, s) for m, s in ranked
                     if underrated_scores.get(m, 0) > 0.7 and m not in chosen_ids]
        und_picks.sort(key=lambda x: x[1], reverse=True)
        for mid, sc in und_picks[:n_underrate]:
            chosen.append((mid, sc))
            chosen_ids.add(mid)

    # 4) Top up if we still have room (in case pools are too small)
    if len(chosen) < top_n:
        for mid, sc in ranked:
            if len(chosen) >= top_n: break
            if mid in chosen_ids: continue
            chosen.append((mid, sc)); chosen_ids.add(mid)

    rows = []
    for mid, sc in chosen[:top_n]:
        rows.append({
            "movie_id": mid,
            "title": movie_lookup.get(mid, {}).get("title", "?"),
            "final_score": float(sc),
            "content_score": cs.get(mid, 0.0),
            "cf_score":      fs.get(mid, 0.0),
            "profile_score": ps.get(mid, 0.0),
            "trending_score":ts.get(mid, 0.0),
            "is_regional":   bool(regional_scores.get(mid, 0)),
        })
    return pd.DataFrame(rows)

def explain(row):
    comps = {
        "content":  row["content_score"],
        "cf":       row["cf_score"],
        "profile":  row["profile_score"],
        "trending": row["trending_score"],
    }
    top = max(comps, key=comps.get)
    reasons = {
        "content":  "Semantically similar to films you watched.",
        "cf":       "Users with similar taste rated this highly.",
        "profile":  "Matches your archetype and stated genre preferences.",
        "trending": "Trending on FilmMosaic right now.",
    }
    extras = ["Regional / under-represented film."] if row["is_regional"] else []
    return reasons[top] + (" " + " ".join(extras) if extras else "")

print("✅ Step 12 done — recommend() + explain() ready")


✅ Step 12 done — recommend() + explain() ready


## STEP 13 — Robust evaluation (leave-one-out + bootstrap CI)


In [13]:
\
NEGATIVES = 99
TOP_K     = 10

# Per-user holdout: one positive (rating >= 4), latest per user
pos = ratings[ratings["rating"] >= 4.0].copy().sort_values(["user_id", "timestamp"])
held_out = pos.groupby("user_id").tail(1).set_index("user_id")[["movie_id"]]
held_out.columns = ["pos_movie"]
print(f"Holdout users: {len(held_out)}")

# Refit the CF model on the train fold (everything except the held-out row)
held_idx = (ratings.reset_index()
                   .merge(held_out.reset_index(),
                          left_on=["user_id", "movie_id"],
                          right_on=["user_id", "pos_movie"])["index"])
train_fold = ratings.drop(held_idx)
print("Train fold size:", len(train_fold), "  (held out:", len(held_out), ")")

old_cf = cf_model
cf_model = train_svd(train_fold)

# Recompute trending and popularity on the train fold only (no leakage).
# We drop (user, movie) pairs that are in the holdout set so the baseline
# truly has no information about the held-out positive.
held_pairs = set(zip(held_out.index, held_out["pos_movie"]))
train_interactions = interactions[
    ~interactions.apply(lambda r: (r["user_id"], r["movie_id"]) in held_pairs, axis=1)
]
print(f"Train-fold interactions: {len(train_interactions)} (of {len(interactions)})")

# Swap to train-fold state for fair evaluation, restore at end of cell.
_production_trending = trending_scores
_production_watched  = watched_pairs
_production_ratings  = ratings
_production_interactions = interactions
watched_pairs = pd.concat([train_fold[["user_id", "movie_id"]],
                           train_interactions[["user_id", "movie_id"]]],
                          ignore_index=True).drop_duplicates()
ratings      = train_fold              # user_watch_history uses train-fold ratings
interactions = train_interactions      # user_watch_history uses train-fold interactions
# trending_scores already swapped below; no other code reads `interactions` after this

def _train_trending_scores():
    df = train_interactions.copy()
    df["recency_w"] = 1.0 / (1.0 + df["age_days"])
    rs = df.groupby("movie_id")["recency_w"].sum().rename("recent_score").reset_index()
    ac = (train_fold.groupby("movie_id").size()
                    .rename("rating_count").reset_index())
    av = (train_fold.groupby("movie_id")["rating"].mean()
                    .rename("avg_rating").reset_index())
    out = movies[["movie_id"]].copy()
    out = out.merge(rs, on="movie_id", how="left")
    out = out.merge(ac, on="movie_id", how="left")
    out = out.merge(av, on="movie_id", how="left")
    out[["recent_score","rating_count","avg_rating"]] = (
        out[["recent_score","rating_count","avg_rating"]].fillna(0)
    )
    cols = ["recent_score","rating_count","avg_rating"]
    out[cols] = MinMaxScaler().fit_transform(out[cols])
    out["t"] = 0.60*out["recent_score"] + 0.25*out["rating_count"] + 0.15*out["avg_rating"]
    out["t"] = MinMaxScaler().fit_transform(out[["t"]])
    return dict(zip(out["movie_id"], out["t"]))

# Popularity baseline = rating count in train fold (no leakage)
_pop_counts = train_fold.groupby("movie_id").size().to_dict()
def popularity_baseline_rank(_uid, pool):
    return sorted(pool, key=lambda m: -_pop_counts.get(m, 0))

# Train-fold trending, swapped into the global so the hybrid uses it too
trending_scores = _train_trending_scores()
def trending_baseline_rank(_uid, pool):
    return sorted(pool, key=lambda m: -trending_scores.get(m, 0.0))

def hybrid_rank(uid, pool):
    sc = hybrid_score(uid, pool)[0]
    return [m for m, _ in sorted(sc.items(), key=lambda x: -x[1])]

def hr_at_k(ranked, held, k=TOP_K):
    return 1.0 if held in ranked[:k] else 0.0

def ndcg_at_k(ranked, held, k=TOP_K):
    for i, m in enumerate(ranked[:k], start=1):
        if m == held:
            return 1.0 / np.log2(i + 1)
    return 0.0

rng = np.random.default_rng(SEED)
all_ids = movies["movie_id"].to_numpy()
user_pool = list(held_out.index)

hr_h, hr_p, hr_t, nd_h, nd_p, nd_t = [], [], [], [], [], []
for uid in tqdm(user_pool, desc="Eval"):
    pos_movie = held_out.at[uid, "pos_movie"]
    neg = rng.choice(all_ids, size=NEGATIVES + 5, replace=False).tolist()
    pool = [pos_movie] + [m for m in neg if m != pos_movie][:NEGATIVES]
    try:
        rh = hybrid_rank(uid, pool)
        rp = popularity_baseline_rank(uid, pool)
        rt = trending_baseline_rank(uid, pool)
    except Exception:
        continue
    hr_h.append(hr_at_k(rh, pos_movie))
    hr_p.append(hr_at_k(rp, pos_movie))
    hr_t.append(hr_at_k(rt, pos_movie))
    nd_h.append(ndcg_at_k(rh, pos_movie))
    nd_p.append(ndcg_at_k(rp, pos_movie))
    nd_t.append(ndcg_at_k(rt, pos_movie))

cf_model = old_cf  # restore production model
trending_scores = _production_trending  # restore production trending
watched_pairs   = _production_watched  # restore production watched_pairs
ratings         = _production_ratings  # restore production ratings
interactions    = _production_interactions  # restore production interactions

def delta_ci(a, b, n_boot=2000, alpha=0.05):
    a = np.asarray(a, float); b = np.asarray(b, float)
    rng_ = np.random.default_rng(SEED)
    diffs = []
    for _ in range(n_boot):
        ai = a[rng_.choice(len(a), len(a), replace=True)]
        bi = b[rng_.choice(len(b), len(b), replace=True)]
        diffs.append(ai.mean() - bi.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(a) - np.mean(b)), float(lo), float(hi)

print(f"\nUsers evaluated: {len(hr_h)}")
print(f"  Hybrid     HR@10    : {np.mean(hr_h):.4f}   NDCG@10: {np.mean(nd_h):.4f}")
print(f"  Popularity HR@10    : {np.mean(hr_p):.4f}   NDCG@10: {np.mean(nd_p):.4f}")
print(f"  Trending   HR@10    : {np.mean(hr_t):.4f}   NDCG@10: {np.mean(nd_t):.4f}")
d_hr_p, lo_hr_p, hi_hr_p = delta_ci(hr_h, hr_p)
d_nd_p, lo_nd_p, hi_nd_p = delta_ci(nd_h, nd_p)
d_hr_t, lo_hr_t, hi_hr_t = delta_ci(hr_h, hr_t)
d_nd_t, lo_nd_t, hi_nd_t = delta_ci(nd_h, nd_t)
print(f"  Δ vs Popular    HR@10 = {d_hr_p:+.4f}  CI [{lo_hr_p:+.4f},{hi_hr_p:+.4f}]"
      f"  NDCG@10 = {d_nd_p:+.4f}  CI [{lo_nd_p:+.4f},{hi_nd_p:+.4f}]")
print(f"  Δ vs Trending   HR@10 = {d_hr_t:+.4f}  CI [{lo_hr_t:+.4f},{hi_hr_t:+.4f}]"
      f"  NDCG@10 = {d_nd_t:+.4f}  CI [{lo_nd_t:+.4f},{hi_nd_t:+.4f}]")
print("✅ Step 13 done")


Holdout users: 240
Train fold size: 6624   (held out: 240 )
Train-fold interactions: 9302 (of 9542)


Eval: 100%|██████████| 240/240 [00:00<00:00, 628.09it/s]



Users evaluated: 240
  Hybrid     HR@10    : 0.3875   NDCG@10: 0.1848
  Popularity HR@10    : 0.2458   NDCG@10: 0.1347
  Trending   HR@10    : 0.2000   NDCG@10: 0.1124
  Δ vs Popular    HR@10 = +0.1417  CI [+0.0583,+0.2208]  NDCG@10 = +0.0500  CI [+0.0007,+0.0953]
  Δ vs Trending   HR@10 = +0.1875  CI [+0.1083,+0.2625]  NDCG@10 = +0.0723  CI [+0.0244,+0.1161]
✅ Step 13 done


## STEP 14 — Save artifacts


In [14]:
\
artifacts = {
    "embeddings.pkl":     movie_embeddings,
    "similarity.pkl":     content_similarity,
    "trending.pkl":       trending_scores,
    "underrated.pkl":     underrated_scores,
    "regional.pkl":       regional_scores,
    "movie_index.pkl":    {"movie_id_to_idx": movie_id_to_idx,
                           "idx_to_movie_id": idx_to_movie_id,
                           "movie_ids": movie_ids_arr.tolist()},
    "profile_matrix.pkl": profile_matrix,
}
for name, obj in artifacts.items():
    with open(ARTIFACT_DIR / name, "wb") as f:
        pickle.dump(obj, f)
    print("  saved", name)
print("\\nAll artifacts in:", ARTIFACT_DIR)
print("✅ Step 14 done")


  saved embeddings.pkl
  saved similarity.pkl
  saved trending.pkl
  saved underrated.pkl
  saved regional.pkl
  saved movie_index.pkl
  saved profile_matrix.pkl
\nAll artifacts in: /Users/johnshrestha/Documents/Recome
✅ Step 14 done


## DEMO — recommendations for sample users (active, early, cold)


In [15]:
\
# Pick one user of each lifecycle stage
def _pick(utype):
    for uid in user_profiles["user_id"]:
        if get_user_type(uid) == utype:
            return int(uid)
    return None

demo_users = []
for ut in ("active_user", "early_user", "cold_start"):
    u = _pick(ut)
    if u is not None:
        demo_users.append(u)

# Plus a few hand-picked horror/crime/drama users
for u in [1, 5, 8, 12, 17, 22]:
    if u not in demo_users:
        demo_users.append(u)

for uid in demo_users:
    prof = profile_lookup.get(uid, {})
    print("=" * 70)
    print(f"User {uid} — archetype={prof.get('archetype','?')}, "
          f"pref={prof.get('preferred_genres','?')}, "
          f"type={get_user_type(uid)}, n_watched={int((watched_pairs['user_id']==uid).sum())}")
    recs = recommend(uid, top_n=8, ensure_regional=2, ensure_underrated=1)
    recs["reason"] = recs.apply(explain, axis=1)
    try:
        from IPython.display import display
        display(recs[["movie_id", "title", "final_score",
                     "content_score", "cf_score", "profile_score",
                     "is_regional", "reason"]])
    except Exception:
        print(recs[["movie_id", "title", "final_score",
                    "content_score", "cf_score", "profile_score",
                    "is_regional", "reason"]].to_string(index=False))
    print()


User 1 — archetype=horror_thriller, pref=Horror|Thriller|Mystery|Music, type=active_user, n_watched=37


,movie_id,title,final_score,content_score,cf_score,profile_score,is_regional,reason
0,11033,Dressed to Kill,0.854439,0.909971,0.710409,1.0,False,Matches your archetype and stated genre preferences.
1,522681,Escape Room,0.830700,0.774223,0.713796,1.0,False,Matches your archetype and stated genre preferences.
2,419430,Get Out,0.830369,0.822162,0.709590,1.0,False,Matches your archetype and stated genre preferences.
3,539,Psycho,0.828279,0.852424,0.667289,1.0,False,Matches your archetype and stated genre preferences.
4,1970,The Grudge,0.813688,0.778953,0.673986,1.0,False,Matches your archetype and stated genre preferences.
5,1338873,Dakini,0.752175,0.696855,0.593205,1.0,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
6,489608,Zhigrana,0.724000,0.584155,0.593205,1.0,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
7,926606,The Home,0.796654,0.682151,0.701235,1.0,False,Matches your archetype and stated genre preferences.



User 9 — archetype=classic_drama, pref=Drama|History|War, type=early_user, n_watched=25


,movie_id,title,final_score,content_score,cf_score,profile_score,is_regional,reason
0,76758,The Flowers of War,0.878587,0.701205,0.841129,1.000000,False,Matches your archetype and stated genre preferences.
1,857,Saving Private Ryan,0.818745,0.660679,0.888545,1.000000,False,Matches your archetype and stated genre preferences.
2,424,Schindler's List,0.793688,0.728941,0.890061,1.000000,False,Matches your archetype and stated genre preferences.
3,613,Downfall,0.787172,0.540591,0.812168,1.000000,False,Matches your archetype and stated genre preferences.
4,659,The Tin Drum,0.745367,0.661163,0.692331,1.000000,False,Matches your archetype and stated genre preferences.
5,495379,Dasdhunga,0.522064,0.548668,0.620651,0.666667,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
6,864734,Sijou,0.493691,0.454093,0.620651,0.666667,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
7,205596,The Imitation Game,0.738987,0.638797,0.860354,0.812500,False,Users with similar taste rated this highly.



User 5 — archetype=horror_thriller, pref=Horror|Thriller|Mystery|Adventure, type=active_user, n_watched=30


...

,movie_id,title,final_score,content_score,cf_score,profile_score,is_regional,reason
0,993,Sleuth,0.926455,1.000000,0.856088,1.000000,False,Semantically similar to films you watched.
1,65754,The Girl with the Dragon Tattoo,0.842037,0.754900,0.817394,1.000000,False,Matches your archetype and stated genre preferences.
2,186,Lucky Number Slevin,0.829943,0.788640,0.899812,0.861538,False,Users with similar taste rated this highly.
3,793,Blue Velvet,0.825792,0.719480,0.812344,1.000000,False,Matches your archetype and stated genre preferences.
4,346685,The Girl on the Train,0.825551,0.827085,0.705873,1.000000,False,Matches your archetype and stated genre preferences.
5,1198434,Antim Sanskar: The Last Ritual,0.687638,0.682855,0.604143,0.861538,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
6,626920,Chakkar,0.635953,0.737922,0.604143,0.666667,True,Semantically similar to films you watched. Regional / under-represented film.
7,414906,The Batman,0.815769,0.764351,0.727931,1.000000,False,Matches your archetype and stated genre preferences.



User 17 — archetype=classic_drama, pref=Drama|History|War|Crime, type=active_user, n_watched=38


,movie_id,title,final_score,content_score,cf_score,profile_score,is_regional,reason
0,857,Saving Private Ryan,0.803830,0.782453,0.850924,0.812500,False,Users with similar taste rated this highly.
1,76758,The Flowers of War,0.802025,0.810775,0.748346,0.812500,False,Trending on FilmMosaic right now.
2,94047,My Way,0.787911,0.840343,0.890166,0.700000,False,Users with similar taste rated this highly.
3,424,Schindler's List,0.773655,0.851847,0.720313,0.812500,False,Semantically similar to films you watched.
4,3175,Barry Lyndon,0.746560,0.767590,0.821492,0.700000,False,Users with similar taste rated this highly.
5,1462704,Karma,0.548173,0.732171,0.589711,0.383333,True,Semantically similar to films you watched. Regional / under-represented film.
6,1150305,Dhoka,0.545027,0.819610,0.589711,0.270833,True,Semantically similar to films you watched. Regional / under-represented film.
7,300671,13 Hours: The Secret Soldiers of Benghazi,0.745568,0.938480,0.692052,0.625000,False,Semantically similar to films you watched.



User 22 — archetype=crime_mystery, pref=Crime|Mystery|Thriller, type=early_user, n_watched=27


,movie_id,title,final_score,content_score,cf_score,profile_score,is_regional,reason
0,993,Sleuth,0.866936,0.979436,0.821100,1.000000,False,Matches your archetype and stated genre preferences.
1,793,Blue Velvet,0.816785,0.859957,0.800313,1.000000,False,Matches your archetype and stated genre preferences.
2,414906,The Batman,0.810424,0.808785,0.817679,1.000000,False,Matches your archetype and stated genre preferences.
3,65754,The Girl with the Dragon Tattoo,0.807777,0.810265,0.768257,1.000000,False,Matches your archetype and stated genre preferences.
4,346685,The Girl on the Train,0.804951,0.853141,0.760087,1.000000,False,Matches your archetype and stated genre preferences.
5,1198434,Antim Sanskar: The Last Ritual,0.625291,0.733285,0.604651,0.812500,True,Matches your archetype and stated genre preferences. Regional / under-represented film.
6,626920,Chakkar,0.569725,0.718206,0.604651,0.666667,True,Semantically similar to films you watched. Regional / under-represented film.
7,186,Lucky Number Slevin,0.769784,0.759899,0.959158,0.812500,False,Users with similar taste rated this highly.


## Regional / under-represented coverage check


In [16]:
\
# Confirm the slate-level diversity injection is doing its job:
# for every user, count how many regional + how many underrated films
# appear in their top-10, on average.
import collections

buckets = collections.defaultdict(lambda: {"regional": 0, "underrated": 0, "n": 0})
for uid in tqdm(user_profiles["user_id"], desc="Coverage"):
    r = recommend(uid, top_n=10, ensure_regional=2, ensure_underrated=1)
    buckets[get_user_type(uid)]["regional"]   += int(r["is_regional"].sum())
    buckets[get_user_type(uid)]["underrated"] += int((r.apply(
        lambda row: underrated_scores.get(row["movie_id"], 0) > 0.7, axis=1)).sum())
    buckets[get_user_type(uid)]["n"]          += 1

print()
print(f"{'user type':<14} {'avg regional / 10':>20} {'avg underrated / 10':>22}")
print("-" * 60)
for ut, b in buckets.items():
    n = max(b["n"], 1)
    print(f"{ut:<14} {b['regional']/n:>20.2f} {b['underrated']/n:>22.2f}")
print()
print("(Target: regional ≥ 2.0 per 10, underrated ≥ 1.0 per 10 — the slate guarantee)")


Coverage: 100%|██████████| 240/240 [00:10<00:00, 23.53it/s]


user type         avg regional / 10    avg underrated / 10
------------------------------------------------------------
active_user                    2.01                   5.68
early_user                     2.02                   5.31

(Target: regional ≥ 2.0 per 10, underrated ≥ 1.0 per 10 — the slate guarantee)
